# Balanced UCI training

Το UCI dataset έχει 74.6% PD bias. Εδώ το **balanc-άρουμε με undersampling των PD subjects** ώστε να έχουμε ίσο αριθμό HC και PD recordings.

Subject-level balancing (όχι recording-level) για να αποφύγουμε leakage.

**Αρχικά**: 64 HC subjects (192 rec.) + 188 PD subjects (564 rec.)
**Μετά**: 64 HC subjects (192 rec.) + 64 PD subjects (192 rec.) = 384 total

In [1]:
import sys, joblib
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, confusion_matrix

from src.features import FEATURE_NAMES

MODELS = Path('../models')

In [2]:
uci = pd.read_csv('../data/uci/pd_speech_features.csv', header=1)
print(f'Original UCI: {len(uci)} recordings')
print(f'  HC: {(uci["class"]==0).sum()} recordings από {uci[uci["class"]==0]["id"].nunique()} subjects')
print(f'  PD: {(uci["class"]==1).sum()} recordings από {uci[uci["class"]==1]["id"].nunique()} subjects')

Original UCI: 756 recordings
  HC: 192 recordings από 64 subjects
  PD: 564 recordings από 188 subjects


In [3]:
# Subject-level undersampling των PD
rng = np.random.default_rng(42)

hc_subjects = uci[uci['class'] == 0]['id'].unique()
pd_subjects = uci[uci['class'] == 1]['id'].unique()

# Διάλεξε ίσο αριθμό PD subjects με τους HC
selected_pd = rng.choice(pd_subjects, size=len(hc_subjects), replace=False)
selected_subjects = np.concatenate([hc_subjects, selected_pd])

uci_balanced = uci[uci['id'].isin(selected_subjects)].reset_index(drop=True)
print(f'\nBalanced UCI: {len(uci_balanced)} recordings')
print(f'  HC: {(uci_balanced["class"]==0).sum()} recordings από {uci_balanced[uci_balanced["class"]==0]["id"].nunique()} subjects')
print(f'  PD: {(uci_balanced["class"]==1).sum()} recordings από {uci_balanced[uci_balanced["class"]==1]["id"].nunique()} subjects')
print(f'  PD ratio: {(uci_balanced["class"]==1).mean()*100:.1f}%')


Balanced UCI: 384 recordings
  HC: 192 recordings από 64 subjects
  PD: 192 recordings από 64 subjects
  PD ratio: 50.0%


## Train balanced model

In [4]:
X = uci_balanced[FEATURE_NAMES]
y = uci_balanced['class'].values
groups = uci_balanced['id'].values

pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
    # class_weight δεν χρειάζεται πια αφού το dataset είναι balanced
])

cv = GroupKFold(10)
y_proba = cross_val_predict(pipe, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print('=== Balanced UCI model ===')
print(f'Accuracy: {accuracy_score(y, y_pred):.3f}')
print(f'F1:       {f1_score(y, y_pred):.3f}')
print(f'MCC:      {matthews_corrcoef(y, y_pred):.3f}')
cm = confusion_matrix(y, y_pred)
print(f'CM: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}')

# Threshold scan
print('\nThreshold scan:')
print('  thr   acc    HC-recall  PD-recall')
for t in [0.40, 0.45, 0.50, 0.55, 0.60]:
    p = (y_proba >= t).astype(int)
    cm = confusion_matrix(y, p)
    hc_r = cm[0,0] / cm[0].sum() if cm[0].sum() else 0
    pd_r = cm[1,1] / cm[1].sum() if cm[1].sum() else 0
    print(f'  {t:.2f}  {accuracy_score(y, p):.3f}  {hc_r:.3f}      {pd_r:.3f}')

pipe.fit(X, y)
joblib.dump(pipe, MODELS / 'uci_balanced.joblib')
print(f'\nSaved uci_balanced.joblib')

=== Balanced UCI model ===
Accuracy: 0.695
F1:       0.696
MCC:      0.391
CM: TN=133, FP=59, FN=58, TP=134

Threshold scan:
  thr   acc    HC-recall  PD-recall
  0.40  0.677  0.557      0.797
  0.45  0.685  0.630      0.740
  0.50  0.695  0.693      0.698
  0.55  0.688  0.760      0.615
  0.60  0.695  0.833      0.557

Saved uci_balanced.joblib


## Comparison with original UCI model

In [5]:
# Reload original
X_full = uci[FEATURE_NAMES]
y_full = uci['class'].values
groups_full = uci['id'].values

pipe_orig = Pipeline([
    ('scaler', RobustScaler()),
    ('clf', RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')),
])
y_proba_orig = cross_val_predict(pipe_orig, X_full, y_full, cv=GroupKFold(10), groups=groups_full, method='predict_proba')[:, 1]
y_pred_orig = (y_proba_orig >= 0.5).astype(int)

print('=== Comparison ===')
print(f'Original UCI (74% PD, class_weight=balanced):')
print(f'  acc={accuracy_score(y_full, y_pred_orig):.3f}, F1={f1_score(y_full, y_pred_orig):.3f}, MCC={matthews_corrcoef(y_full, y_pred_orig):.3f}')
cm = confusion_matrix(y_full, y_pred_orig)
print(f'  HC recall: {cm[0,0]/cm[0].sum():.3f}, PD recall: {cm[1,1]/cm[1].sum():.3f}')

print(f'\nBalanced UCI (50/50 undersampled PD):')
print(f'  acc={accuracy_score(y, y_pred):.3f}, F1={f1_score(y, y_pred):.3f}, MCC={matthews_corrcoef(y, y_pred):.3f}')
cm = confusion_matrix(y, y_pred)
print(f'  HC recall: {cm[0,0]/cm[0].sum():.3f}, PD recall: {cm[1,1]/cm[1].sum():.3f}')

=== Comparison ===
Original UCI (74% PD, class_weight=balanced):
  acc=0.795, F1=0.874, MCC=0.380
  HC recall: 0.323, PD recall: 0.956

Balanced UCI (50/50 undersampled PD):
  acc=0.695, F1=0.696, MCC=0.391
  HC recall: 0.693, PD recall: 0.698
